ESERCIZIO

- Costruire un modello Keras che gestisca testi di lunghezza variabile utilizzando pesi pre-addestrati
- Crea un layer di Embedding per un vocabolario di 5.000 parole, con output a 100 dimensioni e supporto per il masking
- Applica il padding (lunghezza massima 15) a un set di 3 frasi di esempio
- Inizializza il layer con una matrice di pesi casuali (o zeri) e impostalo come non addestrabile (trainable=False), simulando un approccio di Transfer Learning statico.
- Sfida: modifica il codice per rendere i pesi addestrabili e osserva come cambia il numero di parametri totali del modello.

In [1]:
import os

# FASE 0: SETUP DEL MOTORE DI CALCOLO
os.environ["KERAS_BACKEND"] = "torch"

import keras
from keras import layers
import numpy as np

def risolvi_esercizio():
    # 1. Set di 3 frasi di esempio
    frasi_esempio = [
        "Il deep learning è affascinante",
        "Keras rende tutto più semplice",
        "L'elaborazione del linguaggio naturale evolve rapidamente"
    ]
    
    print(f"--- FASE 1: PREPARAZIONE DATI ---")
    # 2. TextVectorization con vocabolario di 5000 parole e padding a 15
    vectorizer = layers.TextVectorization(
        max_tokens=5000,
        output_mode="int",
        output_sequence_length=15 
    )
    
    # Adattiamo il vettorizzatore sulle frasi
    vectorizer.adapt(frasi_esempio)
    
    # Applichiamo il padding alle frasi
    vettori_input = vectorizer(frasi_esempio)
    print(f"Frasi vettorizzate (con padding a 15):\n{vettori_input}\n")
    
    # 3. Costruzione del Modello
    def create_model(trainable=False):
        inputs = layers.Input(shape=(15,), dtype="int32")
        
        # Layer di Embedding: 5000 parole, 100 dimensioni, masking supportato
        embedding_layer = layers.Embedding(
            input_dim=5000,
            output_dim=100,
            mask_zero=True,
            trainable=trainable,
            name="embedding_layer"
        )
        
        x = embedding_layer(inputs)
        x = layers.GlobalAveragePooling1D()(x)
        outputs = layers.Dense(1, activation="sigmoid")(x)
        
        model = keras.Model(inputs, outputs)
        model.compile(optimizer="adam", loss="binary_crossentropy")
        return model

    # --- CASO 1: trainable=False (Statico) ---
    print("--- CASO 1: Pesi NON addestrabili (trainable=False) ---")
    model_static = create_model(trainable=False)
    model_static.summary()
    
    # --- CASO 2: trainable=True (Dinamico) ---
    print("\n--- CASO 2: Pesi addestrabili (trainable=True) ---")
    model_trainable = create_model(trainable=True)
    model_trainable.summary()
    
    # Analizziamo la differenza
    params_static = model_static.count_params()
    trainable_params_static = sum(np.prod(v.shape) for v in model_static.trainable_weights)
    
    params_trainable = model_trainable.count_params()
    trainable_params_dynamic = sum(np.prod(v.shape) for v in model_trainable.trainable_weights)
    
    print("\n--- ANALISI SFIDA ---")
    print(f"Parametri Totali (Statico): {params_static}")
    print(f"Parametri Addestrabili (Statico): {trainable_params_static}")
    print(f"Parametri Totali (Dinamico): {params_trainable}")
    print(f"Parametri Addestrabili (Dinamico): {trainable_params_dynamic}")
    print(f"Differenza parametri addestrabili: {trainable_params_dynamic - trainable_params_static}")

if __name__ == "__main__":
    risolvi_esercizio()

--- FASE 1: PREPARAZIONE DATI ---
Frasi vettorizzate (con padding a 15):
tensor([[13, 16, 11,  2, 17,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [12,  5,  3,  7,  4,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [10, 15,  9,  8, 14,  6,  0,  0,  0,  0,  0,  0,  0,  0,  0]],
       device='cuda:0')

--- CASO 1: Pesi NON addestrabili (trainable=False) ---


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 15)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_layer     │ (None, 15, 100)   │    500,000 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 15)        │          0 │ input_layer[0][0] │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 100)       │          0 │ embedding_layer[… │
│ (GlobalAveragePool… │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 1)         │        101 │ global_average_p… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 500,101 (1.91 MB)

 Trainable params: 101 (404.00 B)

...

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 15)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_layer     │ (None, 15, 100)   │    500,000 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, 15)        │          0 │ input_layer_1[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 100)       │          0 │ embedding_layer[… │
│ (GlobalAveragePool… │                   │            │ not_equal_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 1)         │        101 │ global_average_p… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 500,101 (1.91 MB)

 Trainable params: 500,101 (1.91 MB)

 Non-trainable params: 0 (0.00 B)


--- ANALISI SFIDA ---
Parametri Totali (Statico): 500101
Parametri Addestrabili (Statico): 101
Parametri Totali (Dinamico): 500101
Parametri Addestrabili (Dinamico): 500101
Differenza parametri addestrabili: 500000
